In [6]:
import tensorflow as tf
import os
import os.path
import numpy as np
import cv2
import json
import matplotlib.pyplot as plt
import unicode

In [9]:
import os
import pathlib
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

import unicode

    
def get_label(file_path, class_names):
    parts = tf.strings.split(file_path, os.path.sep)
    return parts[-2] == class_names


def process_path(file_path, class_names, img_shape=(224, 224)):
    label = tf.strings.split(file_path, os.path.sep)
    label = label[-2] == class_names

    img = tf.io.read_file(file_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.convert_image_dtype(img, tf.float32)
    img = tf.image.resize(img, img_shape)
    return img, label


def prepare_for_training(ds, batch_size=32, cache=True, shuffle_buffer_size=1000):
    if cache:
        if isinstance(cache, str):
            ds = ds.cache(cache)
        else:
            ds = ds.cache()

    ds = ds.shuffle(buffer_size=shuffle_buffer_size)
    ds = ds.repeat()
    ds = ds.batch(batch_size)
    ds = ds.prefetch(buffer_size=tf.data.experimental.AUTOTUNE)

    return ds


def load_label(label_path):
    class_names = []
    with open(label_path) as f:
        for line in f:
            line = line.strip()
            class_names.append(line)

    return np.array(class_names)


def show_batch(image_batch, label_batch, class_names):
    size = len(image_batch)
    sub_size = int(size ** 0.5) + 1

    plt.figure(figsize=(10, 10), dpi=80)
    for n in range(size):
        plt.subplot(sub_size, sub_size, n+1)
        plt.subplots_adjust(left=0.125, bottom=0.1, right=0.9, top=0.9, wspace=0.2, hspace=0.5)
        plt.title(class_names[label_batch[n]==True][0].title())
        plt.imshow(image_batch[n])
    plt.show()


def load_data(data_path, label_path, batch=32):
    class_names = load_label(label_path)
    #class_names = unicode.split_syllable_char(load_label(label_path))
    data_dir = pathlib.Path(data_path)
    list_ds = tf.data.Dataset.list_files(str(data_dir / '*/*'))

	# 데이터 확인
    # for f in list_ds.take(5):
    #     print(f.numpy())

    labeled_ds = list_ds.map(lambda x: process_path(x, class_names))
    train_ds = prepare_for_training(labeled_ds, batch_size=batch)

    return train_ds



In [13]:
imagePath = "D:\\data\\letterTest\\image\\23.jpg"
labelPath = "D:\\data\\letterTest\\label\\23.txt"
load_data(imagePath, labelPath)

#tf.data.Dataset.list_files


InvalidArgumentError: Expected 'tf.Tensor(False, shape=(), dtype=bool)' to be true. Summarized data: b'No files matched pattern: D:\\data\\letterTest\\image\\23.jpg\\*\\*'

In [14]:
data_dir = 'D:\\data\\letterTest\\image'
list_ds = tf.data.Dataset.list_files(str(data_dir / '*/*'))
for f in list_ds.take(5):
	print(f.numpy())

TypeError: unsupported operand type(s) for /: 'str' and 'str'

In [17]:
def get_fileList(dirname):
    imagelist = []
    jsonlist = [] 

    try:
        for(path,dir,files) in os.walk(dirname):
            for filename in files:
                split = os.path.splitext(filename)
                if split[1] == '.jpg':
                    full_filename = os.path.join(path, filename)
                    full_jsonname = os.path.join(path,split[0]) + '.json'
                    imagelist.append(full_filename)
                    jsonlist.append(full_jsonname)
        return imagelist, jsonlist
    except PermissionError:
        pass
    
def SaveTrainingData(SrcFiles):
    cnt = 0
    defaultDir = "D:/Anaconda/TrainData"
    imageDir = defaultDir + "/" + "image"
    labelDir = defaultDir + "/" + "image"
    
    imglist, jsonlist = get_fileList(SrcFiles)

    for imgFile, jsonFile in zip(imglist, jsonlist):
        image = cv2.imread(imgFile, cv2.IMREAD_COLOR)
        with open(jsonFile, 'r', encoding='utf-8') as json_file:
            json_data = json.load(json_file)
            charDataRef = json_data["text"]["word"]
            
            for char in charDataRef:
                letterRef = char['letter']

                for letter in letterRef:
                    imageName = imageDir + "/" + str(cnt) + ".jpg"
                    labelName = imageDir + "/" + str(cnt) + ".txt"
                    
                    charVal = letter["value"]

                    #if is_haguel(charVal):
                    with open(labelName , 'w', encoding = 'utf-8') as file :
                        file.write(charVal)
                        
                        x1 = letter['charbox'][0]
                        y1 = letter['charbox'][1]
                        x2 = letter['charbox'][2]
                        y2 = letter['charbox'][3]
    
                        cropped_img = image[y1:y2, x1:x2]
                        cv2.imwrite(imageName, cropped_img)

                        cnt = cnt + 1


In [7]:
# TODO 메모리 외부 파편화 심함

def get_fileList(dirname):
    imagelist = []
    jsonlist = [] 

    try:
        for(path,dir,files) in os.walk(dirname):
            for filename in files:
                split = os.path.splitext(filename)
                if split[1] == '.jpg':
                    full_filename = os.path.join(path, filename)
                    full_jsonname = os.path.join(path,split[0]) + '.json'
                    imagelist.append(full_filename)
                    jsonlist.append(full_jsonname)
        return imagelist, jsonlist
    except PermissionError:
        pass
    

def SaveTrainingData(SrcFiles):
    cnt = 0
    defaultDir = "D:/Anaconda/TrainData"
    imageDir = defaultDir + "/" + "image"
    labelDir = defaultDir + "/" + "label"
    
    imglist, jsonlist = get_fileList(SrcFiles)

    for imgFile, jsonFile in zip(imglist, jsonlist):
        
        
        image = cv2.imread(imgFile, cv2.IMREAD_COLOR)
        with open(jsonFile, 'r', encoding='utf-8') as json_file:
            json_data = json.load(json_file)
            charDataRef = json_data["text"]["word"]
            
            for char in charDataRef:
                letterRef = char['letter']

                for letter in letterRef:
                    imageName = imageDir + "/" + str(cnt) + ".jpg"
                    labelName = imageDir + "/" + str(cnt) + ".txt"
                    
                    charVal = letter["value"]
                    
                    if (not unicode.is_hangul(charVal)):
                        continue

                    #if is_haguel(charVal):
                    with open(labelName , 'w', encoding = 'utf-8') as file :
                        file.write(charVal)
                        
                        x1 = letter['charbox'][0]
                        y1 = letter['charbox'][1]
                        x2 = letter['charbox'][2]
                        y2 = letter['charbox'][3]
    
                        cropped_img = image[y1:y2, x1:x2]
                        cv2.imwrite(imageName, cropped_img)

                        cnt = cnt + 1


In [9]:
if __name__ == "__main__":
    #SaveTrainingData(get_fileList("D:/project/GitTest/han/testData/test"))
    SaveTrainingData("D:\\project\\GitTest\\han\\testData\\test")


In [15]:
import tensorflow as tf
from dataset import load_data

class Trainer:
    def __init__(self, modle, epohcs, batch, loss_fn, optimizer):
        

ModuleNotFoundError: No module named 'dataset'